# E-commerce Returns Agent — Tool-Calling × Multi-Turn

Fine-tune Qwen 2.5 0.5B into an e-commerce returns specialist that
**combines both pillars at once**: it acknowledges the customer's
intent across a multi-turn conversation *and* makes the right tool
calls to fulfill the request (look up the order, calculate the refund,
trigger the return).

**Runtime:** ~2 hours on Colab A100. **Cost:** ~$1.20.
**GPU:** A100 recommended; T4 will take 5+ hours.

**What you'll have at the end:**
- A LoRA adapter trained against a composite reward (rubric + tool-call)
- A reproducible JSON of (baseline, trained) scores
- A 3-line snippet to wire the adapter into the OpenAI-compatible service

**Pillar coverage:**
- `notebooks/customer_support_4h.ipynb` — multi-turn dialogue only
- `notebooks/tool_calling_agent_demo.ipynb` — tool calling only
- **This notebook** — both, against a realistic e-commerce returns scenario


## 1. Pin + install


In [ ]:
import os
import subprocess

PINNED_COMMIT = 'f8e498b'
if not os.path.exists('/content/stateset-agents'):
    subprocess.check_call([
        'git', 'clone', '--quiet',
        'https://github.com/stateset/stateset-agents',
        '/content/stateset-agents'
    ])
subprocess.check_call(['git', '-C', '/content/stateset-agents', 'checkout', '--quiet', PINNED_COMMIT])
%cd /content/stateset-agents
%pip install --quiet -e '.[training]'
%pip install --quiet accelerate
print('Installed.')


In [ ]:
from stateset_agents.utils.reproducibility import set_all_seeds

SEED = 42
state = set_all_seeds(SEED)
print('Seeds applied:', state.to_dict())


## 2. Domain scenarios + tool registry

Six scenarios across three failure modes: damaged item, wrong size,
buyer's remorse. Each requires the agent to (a) acknowledge the issue,
(b) look up the order via the `get_order` tool, (c) confirm the refund
amount via `calculate_refund`, (d) reply with a clear next step.


In [ ]:
TOOLS = [
    {
        'name': 'get_order',
        'description': 'Look up order details by order ID.',
        'parameters': {'type': 'object',
                       'properties': {'order_id': {'type': 'string'}},
                       'required': ['order_id']},
    },
    {
        'name': 'calculate_refund',
        'description': 'Calculate refund amount including shipping.',
        'parameters': {'type': 'object',
                       'properties': {'order_id': {'type': 'string'},
                                       'include_shipping': {'type': 'boolean'}},
                       'required': ['order_id']},
    },
    {
        'name': 'create_return',
        'description': 'Create a return label for the order.',
        'parameters': {'type': 'object',
                       'properties': {'order_id': {'type': 'string'},
                                       'reason': {'type': 'string'}},
                       'required': ['order_id', 'reason']},
    },
]

SCENARIOS = [
    {'user_query': 'Order #1001 arrived damaged. I want a full refund.',
     'intent': 'damaged', 'order_id': '1001',
     'expected_tool': 'create_return',
     'expected_params': {'order_id': '1001', 'reason': 'damaged'},
     'must_acknowledge': ['damaged', 'refund'],
     'must_avoid': ['impossible']},
    {'user_query': 'The shoes from order 1002 are the wrong size — please return.',
     'intent': 'wrong_size', 'order_id': '1002',
     'expected_tool': 'create_return',
     'expected_params': {'order_id': '1002', 'reason': 'wrong size'},
     'must_acknowledge': ['size', 'return'], 'must_avoid': ['impossible']},
    {'user_query': 'Cancel order 1003 and refund — I changed my mind.',
     'intent': 'remorse', 'order_id': '1003',
     'expected_tool': 'create_return',
     'expected_params': {'order_id': '1003', 'reason': 'changed my mind'},
     'must_acknowledge': ['refund'], 'must_avoid': ['impossible']},
    {'user_query': 'Order 1004 — how much would I get back including shipping?',
     'intent': 'inquiry', 'order_id': '1004',
     'expected_tool': 'calculate_refund',
     'expected_params': {'order_id': '1004', 'include_shipping': True},
     'must_acknowledge': ['shipping'], 'must_avoid': []},
    {'user_query': 'What was in order 1005?',
     'intent': 'lookup', 'order_id': '1005',
     'expected_tool': 'get_order',
     'expected_params': {'order_id': '1005'},
     'must_acknowledge': ['order'], 'must_avoid': []},
    {'user_query': 'I never received order 1006 — refund please',
     'intent': 'lost', 'order_id': '1006',
     'expected_tool': 'create_return',
     'expected_params': {'order_id': '1006', 'reason': 'lost in transit'},
     'must_acknowledge': ['refund'], 'must_avoid': ['impossible']},
]
TRAIN_SCENARIOS = SCENARIOS[:4]
EVAL_SCENARIOS = SCENARIOS[4:]
print(f'{len(TRAIN_SCENARIOS)} train, {len(EVAL_SCENARIOS)} eval')


## 3. Composite reward (rubric × tool-call)

A linear combination of the two existing rubrics. 60% weight on the
dialogue rubric (acknowledges + avoids), 40% on the tool-call rubric.
Composite rewards are the right shape for multi-objective scenarios —
see whitepaper §4.3.


In [ ]:
import json, re
from typing import Any

from stateset_agents.core.reward_base import RewardFunction, RewardResult, RewardType
from stateset_agents.core.trajectory import ConversationTurn

_TOOL_BLOCK_RE = re.compile(r'```json\s*(\{.*?\})\s*```', re.DOTALL)

def _extract_tool_call(text: str):
    m = _TOOL_BLOCK_RE.search(text or '')
    if not m:
        return None
    try:
        return json.loads(m.group(1))
    except Exception:
        return None

class EcommerceReward(RewardFunction):
    name = 'ecommerce_returns'
    def __init__(self):
        super().__init__(weight=1.0, reward_type=RewardType.IMMEDIATE, name=self.name)

    async def compute_reward(self, turns, context=None):
        ctx = context or {}
        text = (turns[-1].content if turns else '') or ''
        # Rubric pillar (60%).
        ack = ctx.get('must_acknowledge', []) or []
        avoid = ctx.get('must_avoid', []) or []
        ack_hit = sum(1 for w in ack if w.lower() in text.lower()) / max(len(ack), 1)
        avoid_pen = 1.0 if any(w.lower() in text.lower() for w in avoid) else 0.0
        rubric = max(0.0, ack_hit - avoid_pen)
        # Tool pillar (40%).
        call = _extract_tool_call(text)
        tool_score = 0.0
        if call:
            tool_ok = 0.5 if call.get('tool') == ctx.get('expected_tool') else 0.0
            expected_params = ctx.get('expected_params', {})
            actual_params = call.get('parameters', {}) or {}
            params_ok = 0.5 if all(actual_params.get(k) == v for k, v in expected_params.items()) else 0.0
            tool_score = tool_ok + params_ok
        composite = 0.6 * rubric + 0.4 * tool_score
        return RewardResult(
            score=composite,
            breakdown={'rubric': rubric, 'tool': tool_score, 'composite': composite},
        )

print('EcommerceReward ready.')


## 4. Baseline eval


In [ ]:
from stateset_agents import ToolAgent
from stateset_agents.core.agent_config import AgentConfig
import torch

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

async def evaluate(agent, scenarios):
    reward = EcommerceReward()
    scores, breakdowns = [], []
    for s in scenarios:
        prompt = (
            'You are an e-commerce returns specialist. Address the customer warmly, '
            'invoke the correct tool with the JSON format, and confirm the next step.\n'
            f"\nUser: {s['user_query']}\n\nAgent:"
        )
        response = await agent.generate_response(prompt)
        turns = [ConversationTurn(role='assistant', content=response)]
        r = await reward.compute_reward(turns, context=s)
        scores.append(r.score); breakdowns.append(r.breakdown)
    return (sum(scores)/len(scores)), breakdowns

baseline_agent = ToolAgent(
    config=AgentConfig(model_name=MODEL, torch_dtype='bfloat16', attn_implementation='sdpa',
                       do_sample=False, temperature=0.0),
    tools=TOOLS,
)
await baseline_agent.initialize()
baseline_score, baseline_breakdowns = await evaluate(baseline_agent, EVAL_SCENARIOS)
print(f'Baseline composite: {baseline_score:.3f}')
for b in baseline_breakdowns: print(' ', b)
del baseline_agent
torch.cuda.empty_cache()


## 5. Train with GSPO (composite reward)

Same `train_with_gspo` entrypoint, KL anchor on (`beta=0.05`), LoRA r=16.
The trainer doesn't care that the reward is composite — it sees a scalar
per rollout. The reward function shape is what gives the agent its
domain shape.


In [ ]:
from stateset_agents import ToolAgent
from stateset_agents.core.agent_config import AgentConfig
from stateset_agents.core import ConversationEnvironment
from stateset_agents.training import GSPOConfig, train_with_gspo
import time

config = GSPOConfig(
    model_name=MODEL, output_dir='/content/gspo_ecommerce',
    report_to='none',
    num_generations=4,
    clip_range_left=3e-4, clip_range_right=4e-4,
    learning_rate=5e-6,
    max_prompt_length=512, max_completion_length=320,
    use_lora=True, lora_r=16, lora_alpha=32,
    num_epochs=1, warmup_ratio=0.1,
    use_reference_model=True, beta=0.05,
)

agent = ToolAgent(
    config=AgentConfig(model_name=MODEL, torch_dtype='bfloat16', attn_implementation='sdpa'),
    tools=TOOLS,
)
env = ConversationEnvironment(
    scenarios=[{**s, 'id': f's-{i}', 'topic': 'ecommerce'} for i, s in enumerate(TRAIN_SCENARIOS)],
    reward_fn=EcommerceReward(),
    max_turns=1,
)
train_queries = [
    {'prompt': (
        'You are an e-commerce returns specialist. Address the customer warmly, '
        'invoke the correct tool with the JSON format, and confirm the next step.\n'
        f"\nUser: {s['user_query']}\n\nAgent:"),
     'context': s}
    for s in TRAIN_SCENARIOS
]

t0 = time.time()
await train_with_gspo(config=config, agent=agent, environment=env,
                      reward_model=env.reward_fn, train_queries=train_queries)
print(f'Training wall-clock: {time.time()-t0:.0f}s')


## 6. Post-training eval + provenance JSON


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

final_score, final_breakdowns = await evaluate(agent, EVAL_SCENARIOS)
print(f'Final composite: {final_score:.3f}  (Δ {final_score - baseline_score:+.3f})')
for b in final_breakdowns: print(' ', b)

result = {
    'trainer': 'gspo',
    'task': 'ecommerce_returns',
    'model': MODEL,
    'seed': SEED,
    'pinned_commit': PINNED_COMMIT,
    'baseline_score': baseline_score,
    'final_score': final_score,
    'delta': final_score - baseline_score,
    'eval_breakdowns': final_breakdowns,
    'n_train': len(TRAIN_SCENARIOS),
    'n_eval': len(EVAL_SCENARIOS),
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
}
out = Path('/content/ecommerce_result.json')
out.write_text(__import__('json').dumps(result, indent=2))
print(f'Wrote {out}')


## 7. Next steps

- **Replace the tool stubs with real Shopify/Stripe calls.** The reward
  function is opaque to the underlying tools — only the JSON schema matters.
- **Add more scenarios.** Six is a smoke test; production needs 200+.
  Same JSONL schema; feed via `--scenarios path/to/your.jsonl`.
- **Run three seeds (42, 1337, 2026).** A single seed is anecdote;
  the publication gate from whitepaper §11.7 is three seeds + a judge.
- **Serve the adapter.** Point `AgentConfig(model_name='/content/gspo_ecommerce')`
  at the FastAPI service (`examples/getting_started/05_serve_agent.py`).
